# How to Generate Synthetic Data When Your Data Has Null Values

When you pass data with missing values to SDV, nulls are handled automatically. You don't need to clean or impute them yourself — SDV's preprocessing layer takes care of filling in nulls before the synthesizer trains, and then reintroduces them into the synthetic output.

This cookbook walks you through the default behavior, shows you how to customize it when the defaults aren't enough, and highlights common pitfalls to avoid along the way.

## 1. Loading and exploring the dataset

We'll work with the `null_values_demo_dataset`, a single-table dataset designed for this cookbook. It contains columns with varying amounts of missing values — from 0% to 100% — making it a good stress test for null handling.

**What are the null rates across columns?**

The null rates range from 0% (columns like `ticket_id` and `priority`) all the way to 100% (`customer_notes`, which is entirely null). Some columns have similar null rates — for example, `resolution_time_hours`, `resolved_at`, and `resolution_status` are all around 30%, hinting that they might be null together. We'll investigate this pattern later in Section 7.

Let's visualize the null pattern across the full dataset:

*(Embedded heatmap image will go here)*

## 2. Understanding the metadata

Before fitting a synthesizer, let's look at the **metadata** — SDV's description of the dataset structure. The metadata tells SDV the type of each column (its `sdtype`), which determines how that column's missing values are handled during preprocessing.

Let's visualize what SDV auto-detected:

Each column has an `sdtype` — numerical, categorical, datetime, email, phone, or id. Numerical columns with nulls get their missing values filled with the column mean before training. Categorical columns treat null as just another category. PII columns like email and phone regenerate values using Faker and reintroduce nulls at the original rate.

The key point is that **different column types handle nulls differently**, and the metadata is what drives those decisions.

## 3. Generating synthetic data with default settings

With SDV, generating synthetic data from a dataset with nulls requires no special configuration. Just create a synthesizer, fit it to the data, and sample. The **GaussianCopulaSynthesizer** is a good starting point — it uses classic statistical methods, produces high-quality results, and supports extensive customization.

> _**Enterprise note:** If you're using SDV Enterprise, phone number columns need a custom transformer to avoid a compatibility issue. The `phone_transformer()` helper below returns a fresh `AnonymizedFaker` instance each time it's called, since transformers can only be fit once._

**How do the null rates compare between real and synthetic data?**

The null rates are very close — SDV preserves them out of the box without any null-related configuration. This is because the default mode (`'random'`) reintroduces nulls at roughly the original proportion.

Let's also compare the distribution of a nullable column visually using `get_column_plot()`, which generates an interactive side-by-side comparison:

## 4. Evaluating synthetic data quality

SDV includes a built-in quality evaluation workflow. The `evaluate_quality()` function compares the real and synthetic data across two dimensions: **Column Shapes** (does each column's distribution look similar?) and **Column Pair Trends** (are correlations between columns preserved?). The overall score ranges from 0% to 100%.

It's okay — and even expected — to have a score that is not exactly 100%. A perfect score could actually indicate the synthetic data is too close to the real data.

**How well do null rates match specifically?**

The Quality Report includes `MissingValueSimilarity` as part of Column Shapes. This metric scores how well the synthetic null rates match the real data for each column, from 0.0 (completely different) to 1.0 (identical). Let's compute it per column:

## 5. Controlling how nulls appear in synthetic data

By default, SDV places nulls randomly at roughly the original rate. But what if the *pattern* of missingness matters — for example, unresolved support tickets always have null values for resolution time, resolution date, and resolution status? In that case, you might want the synthesizer to learn *when* values should be null, not just *how often*.

SDV offers three modes for null generation, configured through the `missing_value_generation` parameter:

| Mode | Behavior | Best for |
|------|----------|----------|
| `'random'` (default) | Nulls placed randomly at the original column-level rate | Most use cases — accurate null rates without extra complexity |
| `'from_column'` | The synthesizer learns *when* values should be null based on patterns in other columns | Meaningful missingness — e.g., nulls that are correlated with other features |
| `None` | No nulls are generated for that column | Columns that must be complete in the synthetic output |

You configure these modes using `update_transformers()`. Let's set up synthesizers with `'from_column'` and `None` modes so we can compare all three:

**How do the three modes compare?**

*(Embedded bar chart image will go here)*

A few things stand out in the comparison:

- **`'random'`** closely matches the real null rates across all columns. This is the safest default.
- **`'from_column'`** can sometimes over- or under-produce nulls. Notice `response_time_hours` jumping to 100% null (vs 4.9% real) — this is a known issue with GaussianCopula modeling binary indicators, which we explore in the companion cookbook.
- **`None`** eliminates nulls for numerical and datetime columns, but categorical columns (`category`, `is_escalated`) and PII columns (`customer_email`, `agent_name`, `customer_phone`) still show nulls because they handle missingness differently.

> **Key takeaway:** Use `'random'` (default) unless you specifically need null patterns tied to other columns.

## 6. Cleaning placeholder values before synthesis

A common real-world pattern is using placeholder values like `-1`, `999`, or `"N/A"` instead of actual nulls. SDV does not detect these as missing — it treats them as legitimate data points and will reproduce them in the synthetic output.

Our dataset has two versions of response time that illustrate this:
- `response_time_hours` — uses proper `NaN` for missing values (~5% null)
- `response_time_legacy` — uses `-1.0` as a placeholder (0% null from SDV's perspective)

**Does our dataset have any placeholders?**

*(Embedded image will go here)*

The synthetic data faithfully reproduces `-1` values in `response_time_legacy` because SDV learned them as real data points. Always convert placeholders to `NaN` before fitting:

```python
data = data.replace(-1, np.nan)
data = data.replace('', np.nan)
```

Also watch for empty strings — `pd.isna()` does not flag `""` as missing, so these need to be converted explicitly.

## 7. Preserving correlated null patterns

In our dataset, three columns are always null or non-null together: `resolution_time_hours`, `resolved_at`, and `resolution_status`. This makes sense — they all describe the resolution of a support ticket, so if a ticket isn't resolved, all three fields are missing. This is a **correlated null pattern**.

By default, SDV's `'random'` mode treats each column independently when deciding which synthetic rows should have nulls. This breaks the correlation — you'll see rows where `resolution_time_hours` is null but `resolution_status` is not, which is impossible in the real data.

**Does the default synthesizer preserve this?**

*(Embedded heatmap will go here)*

The null correlation in the real data is perfect (1.0 across all pairs), but the synthetic data shows much weaker correlation. Let's fix this by switching to `'from_column'` mode for the resolution columns, which tells the synthesizer to learn *when* these values should be null:

*(Embedded 3-panel comparison will go here)*

`'from_column'` significantly reduces misalignment but doesn't guarantee perfect correlation — each indicator column is modeled independently. For strict enforcement, use a **`FixedCombinations`** constraint, which locks the observed null/non-null combinations:

```python
from sdv.cag import FixedCombinations

constraint = FixedCombinations(column_names=['resolution_time_hours', 'resolved_at', 'resolution_status'])
synthesizer.add_constraints([constraint])
```

SDV Enterprise also offers **`FixedNullCombinations`**, which enforces null co-occurrence without restricting the permutations of non-null values.

## 8. Customizing null handling per column

You can mix and match null modes across columns using `update_transformers()`. This lets you apply `'from_column'` where missingness is meaningful, `None` where a column must be complete, and leave the default `'random'` for everything else.

The two key parameters on each transformer are:
- **`missing_value_generation`** — controls how nulls reappear in synthetic data (`'random'`, `'from_column'`, or `None`)
- **`missing_value_replacement`** — controls what value replaces nulls during training (default: `'mean'`; alternative: `'random'`, which samples from existing values)

## 9. Recommendations

| Scenario | Recommended mode |
|----------|-----------------|
| Nulls are random / no meaningful pattern | `'random'` (default) |
| Missingness is meaningful or correlated with other features | `'from_column'` |
| Column must have zero nulls in synthetic output | `None` |
| Columns are always null together | `'from_column'` + `FixedCombinations` constraint |

**Checklist before submitting synthetic data:**

1. **Replace placeholders** with `NaN` before fitting — SDV doesn't auto-detect `-1`, `999`, `"N/A"`, or empty strings
2. **Compare null rates** after sampling: `synthetic_data.isnull().mean()` vs `data.isnull().mean()`
3. **Run `evaluate_quality()`** for a full quality assessment including null rate similarity
4. **Test constraints** on a small data subset first — null-related constraint errors are among the most common SDV issues

> **Multi-table note:** As of SDV v1.16, `HMASynthesizer` supports nullable foreign keys — child rows can exist without a parent relationship.